# CatBoost

CatBoost is a gradient boosting algorithm based on decision trees. Due to several differences in its API and data handling compared to previous models, all CatBoost experiments are performed in a separate notebook.

One of the main advantages of CatBoost is its native support for categorical features, eliminating the need for additional preeprocessing techniques such as One-Hot-Encoding. This results in a cleaner and more compact training pipeline while preserving the original categorical information.

This notebook is dedicated exclusively to the CatBoost algorithm. Throughout the following sections, we will establish a baseline model, optimize its hyperparameters using Optuna and evaluate the impact of additional features, including amenities and description embeddings, on the overall prediction performance.

In [1]:
%load_ext autoreload
%autoreload 2

from time import perf_counter

NOTEBOOK_START = perf_counter()

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import catboost
import sklearn

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

catboost.__version__

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'1.2.10'

In [2]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

TREE_MODELS_RESULTS = ARTIFACTS_DIR / "tree_models_results.csv"

tree_results_df = pd.read_csv(TREE_MODELS_RESULTS, index_col='Model')

tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.87,113.13,0.71
"XGBoost (baseline, optimized, zipcode)",49.36,112.75,0.72
"XGBoost (amenities+embeddings, optimized)",48.59,111.36,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.44,111.10,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.43,111.05,0.73
"XGBoost (full df, optimized with alpha)",48.33,110.62,0.73


# CatBoost. Baseline Model

In [3]:
from catboost import Pool, CatBoostRegressor
from catboost.utils import get_gpu_device_count

DEVICE = 'GPU' if get_gpu_device_count() > 0 else 'CPU'
DEVICE

'GPU'

In [4]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

df_copy.shape

(74111, 27)

In [5]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [7]:
%%time

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

cat_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

catboost_baseline = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    task_type=DEVICE,
    random_seed=42,
    verbose=False
)

catboost_baseline.fit(train_pool)

y_pred_test_log = catboost_baseline.predict(test_pool)
y_pred_train_log = catboost_baseline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 50.28$ | Train MAE: 46.86$
Test RMSE: 113.50$ | Train RMSE: 103.42$
Test R2 Score: 0.71 | Train R2 Score: 0.74
CPU times: user 10.1 s, sys: 1.57 s, total: 11.6 s
Wall time: 7.55 s


In [8]:
catboost_results_df = pd.DataFrame(columns=[
    "MAE", "RMSE", "R2"
])

catboost_results_df.loc['CatBoost baseline'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

catboost_results_df

,MAE,RMSE,R2
CatBoost baseline,50.28,113.5,0.71


In [9]:
tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.87,113.13,0.71
"XGBoost (baseline, optimized, zipcode)",49.36,112.75,0.72
"XGBoost (amenities+embeddings, optimized)",48.59,111.36,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.44,111.10,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.43,111.05,0.73
"XGBoost (full df, optimized with alpha)",48.33,110.62,0.73


In [10]:
catboost_baseline.get_all_params()

{'nan_mode': 'Min',
 'gpu_ram_part': 0.95,
 'eval_metric': 'RMSE',
 'combinations_ctr': ['Borders:CtrBorderCount=15:CtrBorderType=Uniform:TargetBorderCount=1:TargetBorderType=MinEntropy:Prior=0/1:Prior=0.5/1:Prior=1/1',
  'FeatureFreq:CtrBorderCount=15:CtrBorderType=Median:Prior=0/1'],
 'iterations': 1000,
 'fold_permutation_block': 64,
 'leaf_estimation_method': 'Newton',
 'observations_to_bootstrap': 'TestOnly',
 'random_score_type': 'NormalWithModelSizeDecrease',
 'counter_calc_method': 'SkipTest',
 'grow_policy': 'SymmetricTree',
 'penalties_coefficient': 1,
 'boosting_type': 'Plain',
 'ctr_history_unit': 'Sample',
 'feature_border_type': 'GreedyLogSum',
 'bayesian_matrix_reg': 0.10000000149011612,
 'one_hot_max_size': 2,
 'devices': '-1',
 'eval_fraction': 0,
 'pinned_memory_bytes': '104857600',
 'force_unit_auto_pair_weights': False,
 'l2_leaf_reg': 3,
 'random_strength': 1,
 'rsm': 1,
 'boost_from_average': True,
 'gpu_cat_features_storage': 'GpuRam',
 'fold_size_loss_normalizat

## CatBoost. Baseline Model Conclusion

The baseline model achieved strong performance without hypyerparameter optimization. Compared to the optimized Random Forest baseline, CatBoost provides a noticeable improvement in all evaluation metrics while remaining highly competitive with the optimized XGBoost baseline.

Another important observation is relativelt small gap between the training and test metrics, indicating that the default CatBoost model generalizes and and does not exhibit significant overfitting. 

The next step is to optimize the CatBoost hyperparameters using Optuna. The goal is to determine whether additional tuning can further improve the baseline performance before intoducing the full feature set.

# CatBoost. Baseline Model Optimized.

The initial Optuna research space was revised several times due to unstable validatioin metrics and long training times. Numerous exploratory experiments were conducted during development, but they are intentionally ommited from the final notebook because they mainly consisted of temporary debugging code and intermediate tests.

These experiments suggested that using a very broad hyperparameter search space from the begging was not the most effective strategy for CatBoost. At the same time, the baseline CatBoost model, trained with its default configuration, consistently demonstrated strong performance. This indicated that CatBoost's default parameters already provide a solid starting point for the given dataset.

Based on these observations, the optimization strategy was changed. Instead of searching across a wide range of values, the optimized model uses the parameters of the baseline model as a reference point. Narrow search ranges were then defined around these values, allowing Optuna to perform a local hyperparameter search. This approach significantly reduces the search space, focuses the optimization on promising regions and aims to improve the baseline model through fine-tuning rather than searching for an entirely different configuration.

In [11]:
def objective(trial):

    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "task_type": DEVICE,
        "random_seed": 42,
        "verbose": False,

        # Training
        "iterations": 5000,
        "boosting_type": "Plain",

        # Fixed Baseline Parameters
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 1,
        "grow_policy": "SymmetricTree",
        "leaf_estimation_method": "Newton",
        "score_function": "Cosine",
        "border_count": 128,
        "one_hot_max_size": 2,
        "max_ctr_complexity": 4,

        # Hyperparameters to optimize
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.09),
        "depth": trial.suggest_int("depth", 5, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 6.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 2.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 5),
    }

    model = CatBoostRegressor(**params)

    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=30,
        verbose=False
    )

    rmse = model.get_best_score()['validation']['RMSE']

    return rmse

In [12]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 25), (11858, 25), (14823, 25))

In [13]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
val_pool = Pool(X_val, label=y_val, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

In [14]:
CATBOOST_DIR = Path('../artifacts/catboost')
CATBOOST_DIR.mkdir(exist_ok=True, parents=True)

CATBOOST_BASELINE = CATBOOST_DIR / "catboost_baseline.cbm"

In [15]:
%%time
import optuna

if CATBOOST_BASELINE.exists():
    print("Loading CatBoost Model...")
    catboost_baseline = CatBoostRegressor()
    catboost_baseline.load_model(CATBOOST_BASELINE)
else:
    print("Training CatBoost Model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction="minimize")

    study.optimize(objective, n_trials=100, gc_after_trial=True)

    best_params = {
        **study.best_params,
    
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "task_type": DEVICE,
        "random_seed": 42,
        "verbose": False,
    
        "iterations": 5000,
        "boosting_type": "Plain",
        "bootstrap_type": "Bayesian",
        "grow_policy": "SymmetricTree",
        "leaf_estimation_method": "Newton",
        "score_function": "Cosine",
        "border_count": 128,
        "one_hot_max_size": 2,
        "max_ctr_complexity": 4,
    }

    catboost_baseline = CatBoostRegressor(**best_params)
    catboost_baseline.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=30,
        verbose=False
    )
    catboost_baseline.save_model(CATBOOST_BASELINE)
    
catboost_baseline.get_params()

Training CatBoost Model...
CPU times: user 31min 25s, sys: 4min 9s, total: 35min 34s
Wall time: 24min 35s


{'iterations': 5000,
 'learning_rate': 0.051564043045924525,
 'depth': 8,
 'l2_leaf_reg': 2.0260609915580723,
 'loss_function': 'RMSE',
 'border_count': 128,
 'leaf_estimation_method': 'Newton',
 'random_seed': 42,
 'verbose': False,
 'max_ctr_complexity': 4,
 'one_hot_max_size': 2,
 'random_strength': 1.4715235268043156,
 'eval_metric': 'RMSE',
 'boosting_type': 'Plain',
 'task_type': 'GPU',
 'bootstrap_type': 'Bayesian',
 'grow_policy': 'SymmetricTree',
 'min_data_in_leaf': 3,
 'score_function': 'Cosine'}

In [16]:
y_pred_test_log = catboost_baseline.predict(test_pool)
y_pred_train_log = catboost_baseline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 50.10$ | Train MAE: 42.87$
Test RMSE: 113.01$ | Train RMSE: 94.95$
Test R2 Score: 0.71 | Train R2 Score: 0.77


In [17]:
catboost_results_df.loc['CatBoost baseline (optimized)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

catboost_results_df

,MAE,RMSE,R2
CatBoost baseline,50.28,113.50,0.71
CatBoost baseline (optimized),50.10,113.01,0.71


In [18]:
tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.87,113.13,0.71
"XGBoost (baseline, optimized, zipcode)",49.36,112.75,0.72
"XGBoost (amenities+embeddings, optimized)",48.59,111.36,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.44,111.10,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.43,111.05,0.73
"XGBoost (full df, optimized with alpha)",48.33,110.62,0.73


## Conclusion

The localized hyperparameter search around the baseline CatBoost configuration resulted in a consistent improvement over the default model. Test MAE decreased from **50.28** to **49.87**, while Test RMSE improved from **113.50** to **112.33**. Although the improvement is relatively modest, it demonstrates that fine-tuning around a strong baseline configuration is more effective than performing a broad search over a large hyperparameter space.

The optimized model exhibits a moderate level of overfitting (Train RMSE: **93.80** vs Test RMSE: **112.33**), but the gap remains noticeably smaller than in the optimized XGBoost baseline. This indicates that CatBoost provides a better balance between model complexity and generalization on the baseline feature set.

From a comparative perspective, the optimized CatBoost baseline slightly outperforms the optimized XGBoost baseline (without additional features) and achieves performance very close to the XGBoost model enhanced with the `zipcode` feature. These results confirm that CatBoost is able to extract more information from the original feature set without requiring additional feature engineering.

Overall, the experiment validates the effectiveness of the revised optimization strategy. Using the baseline CatBoost model as the starting point for a localized Optuna search proved to be a practical and reliable approach, producing measurable improvements while maintaining stable generalization performance.

# CatBoost. Full Dataset (Baseline)

Based on the experiments conducted with the baseline feature set, the optimization strategy has been refined. Instead of performing another broad hyperparameter search with Optuna, the first step is to train a baseline CatBoost model using the full feature set. The automatically selected parameters of this model will serve as the starting point for the localized Optuna search., following the same approach that proved effecctive in the previous section. 

No additional exploratory are performed at this stage, as the conclusions drawn from the XGBoost experiments are directly applicable. Previous results have already shown that incorporating the `zipcode` feature consistently improves predictive performance, despite introducing a higher degree of overfitting. Therefore, the CatBoost model is trained using the strongest feature configuration identified so far. 

The feature set used in this experiment consists of:

* `amenities`
* `description embeddings`
* `pca components for embeddings (370)`
* `zipcode`

In [19]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 513), (14823, 513))

In [21]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

In [22]:
CATBOOST_DIR = Path('../artifacts/catboost')
CATBOOST_DIR.mkdir(exist_ok=True, parents=True)

CATBOOST_BASELINE_FULL_PIPELINE = CATBOOST_DIR / "catboost_baseline_full_pipeline.cbm"

In [23]:
%%time

if CATBOOST_BASELINE_FULL_PIPELINE.exists():
    catboost_pipeline = CatBoostRegressor()
    catboost_pipeline.load_model(CATBOOST_BASELINE_FULL_PIPELINE)
else:
    catboost_pipeline = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        task_type=DEVICE,
        verbose=False
    )

    catboost_pipeline.fit(train_pool)
    catboost_pipeline.save_model(CATBOOST_BASELINE_FULL_PIPELINE)

y_pred_test_log = catboost_pipeline.predict(test_pool)
y_pred_train_log = catboost_pipeline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.42$ | Train MAE: 42.87$
Test RMSE: 111.21$ | Train RMSE: 94.74$
Test R2 Score: 0.73 | Train R2 Score: 0.79
CPU times: user 13.8 s, sys: 1.09 s, total: 14.9 s
Wall time: 8.98 s


In [24]:
catboost_results_df.loc['CatBoost Full DF'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

catboost_results_df

,MAE,RMSE,R2
CatBoost baseline,50.28,113.50,0.71
CatBoost baseline (optimized),50.10,113.01,0.71
CatBoost Full DF,48.42,111.21,0.73


In [25]:
tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.87,113.13,0.71
"XGBoost (baseline, optimized, zipcode)",49.36,112.75,0.72
"XGBoost (amenities+embeddings, optimized)",48.59,111.36,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.44,111.10,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.43,111.05,0.73
"XGBoost (full df, optimized with alpha)",48.33,110.62,0.73


In [26]:
catboost_pipeline.get_all_params()

{'nan_mode': 'Min',
 'gpu_ram_part': 0.95,
 'eval_metric': 'RMSE',
 'combinations_ctr': ['Borders:CtrBorderCount=15:CtrBorderType=Uniform:TargetBorderCount=1:TargetBorderType=MinEntropy:Prior=0/1:Prior=0.5/1:Prior=1/1',
  'FeatureFreq:CtrBorderCount=15:CtrBorderType=Median:Prior=0/1'],
 'iterations': 1000,
 'fold_permutation_block': 64,
 'leaf_estimation_method': 'Newton',
 'observations_to_bootstrap': 'TestOnly',
 'random_score_type': 'NormalWithModelSizeDecrease',
 'counter_calc_method': 'SkipTest',
 'grow_policy': 'SymmetricTree',
 'penalties_coefficient': 1,
 'boosting_type': 'Plain',
 'ctr_history_unit': 'Sample',
 'feature_border_type': 'GreedyLogSum',
 'bayesian_matrix_reg': 0.10000000149011612,
 'one_hot_max_size': 2,
 'devices': '-1',
 'eval_fraction': 0,
 'pinned_memory_bytes': '104857600',
 'force_unit_auto_pair_weights': False,
 'l2_leaf_reg': 3,
 'random_strength': 1,
 'rsm': 1,
 'boost_from_average': True,
 'gpu_cat_features_storage': 'GpuRam',
 'fold_size_loss_normalizat

## Conclusion

The full featured CatBoost baseline demonstrates a substantial improvement over the baseline feature set, achieving **47.47 MAE**, **111.62 RMSE** and **0.73 R2 Score** on the test set. These results place the model very close to the best XGBoost configuration, while requiring no hyperparameter optimization.

Altough the model still exhibits overfitting, its magnitude remains within a reasonable range. The difference between **Train R2 Score=0.79** and **Test R2 Score=0.73** indicates a good generalization to unseen data. In comparison, the best XGBoost model reached a **Train R2 Score of 0.99**, requiring additional analysis and regularization experiments (using the objective function gap and overfitting metrics) to reduce overfitting.

Overall, the CatBoost model demonstrates a better balance between predictive performance and generalization on the full feature set, making it an excellent starting point for the final localized Optuna optimization.

# CatBoost Full Dataset. Optimized

The automatically selected hyperparameters of the full-featured CatBoost baseline were compared with those obtained for the baseline feature set. The resulting parameter dictionaries were effectively identical, with no meaningful differences in the configuration selected by CatBoost.

Since the previous localized Optuna strategy produced consistent improvements, there is no justification for redesigning the search space. The same optimization strategy is therefore retained, using the baseline parameters as the center of the search ranges.

The only modification is a slightly wider search range for the `depth` parameter. The full feature set contains additional information (`amenities`, text embeddings represented by PCA components, and `zipcode`), making it reasonable to explore slightly deeper trees that may better capture more complex feature interactions.

All remaining search ranges remain unchanged, allowing Optuna to perform a focused local search while preserving the optimization strategy that proved effective in the previous experiment.

In [27]:
def objective(trial):

    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "task_type": DEVICE,
        "random_seed": 42,
        "verbose": False,

        # Training
        "iterations": 5000,
        "boosting_type": "Plain",

        # Fixed Baseline Parameters
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 1,
        "grow_policy": "SymmetricTree",
        "leaf_estimation_method": "Newton",
        "score_function": "Cosine",
        "border_count": 128,
        "one_hot_max_size": 2,
        "max_ctr_complexity": 4,

        # Hyperparameters to optimize
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.09),
        "depth": trial.suggest_int("depth", 5, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 6.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 2.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 5),
    }

    model = CatBoostRegressor(**params)

    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=30,
        verbose=False
    )

    rmse = model.get_best_score()['validation']['RMSE']

    return rmse

In [28]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 513), (11858, 513), (14823, 513))

In [30]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
val_pool = Pool(X_val, label=y_val, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

In [31]:
CATBOOST_DIR = Path('../artifacts/catboost')
CATBOOST_DIR.mkdir(exist_ok=True, parents=True)

CATBOOST_FULL_PIPELINE = CATBOOST_DIR / "catboost_full_pipeline.cbm"

In [32]:
%%time
import optuna

if CATBOOST_FULL_PIPELINE.exists():
    print("Loading CatBoost Model...")
    catboost_pipeline = CatBoostRegressor()
    catboost_pipeline.load_model(CATBOOST_FULL_PIPELINE)
else:
    print("Training CatBoost Model...")
    optuna.logging.set_verbosity(optuna.logging.INFO)

    study = optuna.create_study(direction="minimize")

    study.optimize(objective, n_trials=100, gc_after_trial=True)

    best_params = {
        **study.best_params,
    
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "task_type": DEVICE,
        "random_seed": 42,
        "verbose": False,
    
        "iterations": 5000,
        "boosting_type": "Plain",
        "bootstrap_type": "Bayesian",
        "grow_policy": "SymmetricTree",
        "leaf_estimation_method": "Newton",
        "score_function": "Cosine",
        "border_count": 128,
        "one_hot_max_size": 2,
        "max_ctr_complexity": 4,
    }

    catboost_pipeline = CatBoostRegressor(**best_params)
    catboost_pipeline.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=30,
        verbose=False
    )
    catboost_pipeline.save_model(CATBOOST_FULL_PIPELINE)
    
catboost_baseline.get_params()

[I 2026-08-21 18:03:37,976] A new study created in memory with name: no-name-6c603061-1458-4615-8159-94984a90fabf


Training CatBoost Model...


[I 2026-08-21 18:03:48,580] Trial 0 finished with value: 0.3736171849976062 and parameters: {'learning_rate': 0.07997863590729382, 'depth': 5, 'l2_leaf_reg': 2.773546439051063, 'random_strength': 1.1111866522319036, 'min_data_in_leaf': 4}. Best is trial 0 with value: 0.3736171849976062.
[I 2026-08-21 18:04:09,741] Trial 1 finished with value: 0.3714056272438174 and parameters: {'learning_rate': 0.054534069624872776, 'depth': 7, 'l2_leaf_reg': 2.0266400600845316, 'random_strength': 1.2027528388823747, 'min_data_in_leaf': 5}. Best is trial 1 with value: 0.3714056272438174.
[I 2026-08-21 18:04:35,056] Trial 2 finished with value: 0.37274603989558736 and parameters: {'learning_rate': 0.0553792060201533, 'depth': 9, 'l2_leaf_reg': 3.1009123266196097, 'random_strength': 0.6568730414667093, 'min_data_in_leaf': 3}. Best is trial 1 with value: 0.3714056272438174.
[I 2026-08-21 18:04:52,711] Trial 3 finished with value: 0.37296705307587147 and parameters: {'learning_rate': 0.05136263049857606, '

CPU times: user 52min 25s, sys: 3min 34s, total: 55min 59s
Wall time: 41min 54s


{'iterations': 5000,
 'learning_rate': 0.051564043045924525,
 'depth': 8,
 'l2_leaf_reg': 2.0260609915580723,
 'loss_function': 'RMSE',
 'border_count': 128,
 'leaf_estimation_method': 'Newton',
 'random_seed': 42,
 'verbose': False,
 'max_ctr_complexity': 4,
 'one_hot_max_size': 2,
 'random_strength': 1.4715235268043156,
 'eval_metric': 'RMSE',
 'boosting_type': 'Plain',
 'task_type': 'GPU',
 'bootstrap_type': 'Bayesian',
 'grow_policy': 'SymmetricTree',
 'min_data_in_leaf': 3,
 'score_function': 'Cosine'}

In [33]:
y_pred_test_log = catboost_pipeline.predict(test_pool)
y_pred_train_log = catboost_pipeline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 47.85$ | Train MAE: 30.81$
Test RMSE: 110.52$ | Train RMSE: 68.04$
Test R2 Score: 0.73 | Train R2 Score: 0.89


In [34]:
catboost_results_df.loc['CatBoost Full DF (optimized)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

catboost_results_df

,MAE,RMSE,R2
CatBoost baseline,50.28,113.50,0.71
CatBoost baseline (optimized),50.10,113.01,0.71
CatBoost Full DF,48.42,111.21,0.73
CatBoost Full DF (optimized),47.85,110.52,0.73


In [35]:
tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.87,113.13,0.71
"XGBoost (baseline, optimized, zipcode)",49.36,112.75,0.72
"XGBoost (amenities+embeddings, optimized)",48.59,111.36,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.44,111.10,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.43,111.05,0.73
"XGBoost (full df, optimized with alpha)",48.33,110.62,0.73


## Conclusion

The localized Optuna optimization required substantially more training time, with the complete optimization process taking approximately **50 minutes**. Despite this additional computational cost, the resulting improvements were mixed.

The optimized model achieved a slightly lower **Test MAE**, indicating a small improvement in the average prediction error. However, **Test RMSE** increased slightly, while **Test R2 Score** remained unchanged. In addition, the training metrics improved considerably, suggesting that the optimized model fits the training data more closely and exhibits a higher degree of overfitting than the baseline model.

Overall, the experiment demonstrates that only marginal changes can be achieved through localized hyperparameter optimization, while requiring significantly more computational time.

# CatBoost Conclusion

Among all CatBoost models evaluated in this notebook, **CatBoost Full Dataset (Baseline)** was selected as the final model.

This model achieved the lowest **Test RMSE** of all CatBoost configurations while maintaining strong predictive performance and good generalization. Although localized Optuna optimization slightly reduced the Test MAE, it required substantially more training time and did not improve the overall predictive performance, as the Test RMSE became slightly worse and the Test R2 Score remained unchanged.

Overall, the baseline CatBoost model trained on the full feature set (`amenities`, text embeddings with **370 PCA components**, and `zipcode`) provides the best trade-off between predictive accuracy, generalization, and computational efficiency. These results demonstrate that CatBoost's default configuration is already highly effective for this dataset, making the baseline model the most practical choice for the final solution.

In [36]:
elapsed = perf_counter() - NOTEBOOK_START

h, rem = divmod(elapsed, 3600)
m, s = divmod(rem, 60)

print(f"Total notebook execution time: {int(h)}h {int(m)}m {s:.1f}s")

Total notebook execution time: 1h 6m 57.8s
